# Exp 20: Coarse dropout on non-VIP images

Card: [exp20_coarse_dropout_non_vip.md](../../docs/experiments/exp20_coarse_dropout_non_vip.md)

In [ ]:
import matplotlib.pyplot as plt

from core.training.config import ROOT
from core.training.config import VIP
from core.training.cv import CVResult
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import pr_curves, save_figures, use_style

N_SAMPLE = 16  # exp19 sample
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp20_coarse_dropout_non_vip"

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "coarse dropout": "#2a78d6"}
use_style()

## Training

In [ ]:
results = {"exp16": CVResult.load(EXP16)}
results["coarse dropout"] = CVResult.load(CONFIG)
figures = {}

## Grad-CAM Cat, Dog: exp16 vs coarse dropout

In [ ]:
cams = {"exp16": GradCamFigures(EXP16), "coarse dropout": GradCamFigures(CONFIG)}
source = "exp16"  # exp19 rows drawn from exp16 OOF
for cls in ["Cat", "Dog"]:
    rows = cams[source].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = cams["coarse dropout"].positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)